# Vesuvius V3 FIXED Inference

## Uses InstanceNorm3d (standard PyTorch) to match V3 training

**Issue Found**: V3 training used SafeInstanceNorm3d but loaded V2 weights (GroupNorm) with `strict=False`.
This caused normalization layer weights to be incorrectly initialized.

**Fix**: Use standard `nn.InstanceNorm3d` which has compatible parameter names with the saved weights.

In [ ]:
!pip install imagecodecs connected-components-3d tifffile -q

In [ ]:
import os
import sys
import gc
import zipfile
import warnings
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from scipy import ndimage
from skimage.morphology import remove_small_objects

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("cc3d not found, using scipy for connected components")

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    TEST_IMAGES_DIR = Path("/kaggle/input/vesuvius-challenge-surface-detection/test_images")
    TEST_CSV = Path("/kaggle/input/vesuvius-challenge-surface-detection/test.csv")
    
    # UPDATE THIS PATH to your V3 trained model checkpoint
    CHECKPOINT_PATH = Path("/kaggle/input/vesuvius-esembled-model/pytorch/v3/1/checkpoints_v3/best_model.pth")
    
    OUTPUT_DIR = Path("/kaggle/working/submission_masks")
    SUBMISSION_ZIP = Path("/kaggle/working/submission.zip")
    
    # Architecture - MUST match V3 training
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)
    FEATURES: List[int] = [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # Inference settings
    OVERLAP: float = 0.5
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # Post-processing
    FINAL_THRESHOLD: float = 0.55
    MIN_COMPONENT_SIZE: int = 25

cfg = Config()
cfg.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Device: {cfg.DEVICE}")
if cfg.DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} (CUDA {props.major}.{props.minor})")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")


# =============================================================================
# FIRST: Inspect checkpoint to understand what's inside
# =============================================================================
print("\n" + "="*70)
print("CHECKPOINT INSPECTION")
print("="*70)

checkpoint = torch.load(cfg.CHECKPOINT_PATH, map_location='cpu', weights_only=False)
print(f"Checkpoint keys: {list(checkpoint.keys())}")

if 'best_score' in checkpoint:
    print(f"Best score: {checkpoint['best_score']:.4f}")
if 'best_epoch' in checkpoint:
    print(f"Best epoch: {checkpoint['best_epoch']}")
if 'epoch' in checkpoint:
    print(f"Saved at epoch: {checkpoint['epoch']}")

state_dict = checkpoint.get('model_state_dict', checkpoint)

# Check what type of normalization the weights have
norm_keys = [k for k in state_dict.keys() if 'norm' in k.lower() or 'running' in k.lower()]
print(f"\nNormalization-related keys (first 10): {norm_keys[:10]}")

# Check for specific patterns
has_running_mean = any('running_mean' in k for k in state_dict.keys())
has_num_batches = any('num_batches_tracked' in k for k in state_dict.keys())
print(f"\nHas running_mean (BatchNorm/InstanceNorm): {has_running_mean}")
print(f"Has num_batches_tracked: {has_num_batches}")

# Sample some keys
print(f"\nFirst 20 keys in state_dict:")
for i, k in enumerate(list(state_dict.keys())[:20]):
    print(f"  {k}: {state_dict[k].shape}")

print("="*70)

In [ ]:
# =============================================================================
# MODEL - Try with standard InstanceNorm3d first
# =============================================================================

class ConvBlock(nn.Module):
    """3D convolution + InstanceNorm3d + LeakyReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            nn.InstanceNorm3d(out_ch, affine=True),  # Standard PyTorch InstanceNorm3d
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )

    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)

    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    """ResEncUNet with InstanceNorm3d."""

    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()

        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]

        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)

        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)

            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())

            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2, bias=True))

        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()

        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))

        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)

        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(4, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1, bias=True))

    def forward(self, x):
        enc_features = []

        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)

        enc_features = enc_features[::-1]
        x = enc_features[0]

        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        return self.final(x)


# =============================================================================
# Create model and try to load weights
# =============================================================================
print("\nCreating model with InstanceNorm3d...")
model = ResEncUNet3D(
    features=cfg.FEATURES,
    n_blocks=cfg.N_BLOCKS,
    use_scse=cfg.USE_SCSE,
    use_deep_supervision=cfg.USE_DEEP_SUPERVISION
)

# Print model keys for comparison
model_keys = list(model.state_dict().keys())
print(f"\nModel expects {len(model_keys)} keys")
print(f"Checkpoint has {len(state_dict)} keys")

# Clean checkpoint keys
cleaned_state = {}
for k, v in state_dict.items():
    key = k.replace('module.', '').replace('_orig_mod.', '')
    cleaned_state[key] = v

# Find matching and missing keys
model_key_set = set(model_keys)
ckpt_key_set = set(cleaned_state.keys())

matching = model_key_set & ckpt_key_set
missing_in_ckpt = model_key_set - ckpt_key_set
extra_in_ckpt = ckpt_key_set - model_key_set

print(f"\nMatching keys: {len(matching)}")
print(f"Missing in checkpoint: {len(missing_in_ckpt)}")
print(f"Extra in checkpoint: {len(extra_in_ckpt)}")

if missing_in_ckpt:
    print(f"\nMissing keys (first 10): {list(missing_in_ckpt)[:10]}")
if extra_in_ckpt:
    print(f"\nExtra keys (first 10): {list(extra_in_ckpt)[:10]}")

# Try loading
missing, unexpected = model.load_state_dict(cleaned_state, strict=False)
print(f"\nLoad result: {len(missing)} missing, {len(unexpected)} unexpected")

model = model.to(cfg.DEVICE).eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

In [ ]:
# =============================================================================
# NORMALIZATION & HELPERS
# =============================================================================
def normalize_volume(volume: np.ndarray) -> np.ndarray:
    """Simple z-score normalization."""
    return (volume.astype(np.float32) - volume.mean()) / (volume.std() + 1e-8)


def simplified_postprocess(pred_prob, threshold=0.55, min_component_size=25):
    print(f"  Initial FG% (threshold {threshold}): {100 * (pred_prob > threshold).mean():.3f}%")
    binary = (pred_prob > threshold).astype(bool)
    cleaned = remove_small_objects(binary, min_size=min_component_size, connectivity=26)
    final = cleaned.astype(np.uint8)
    print(f"  Final FG% (after removing < {min_component_size}): {100 * final.mean():.3f}%")
    return final


@torch.no_grad()
def sliding_window_inference(model, volume, patch_size=(128, 128, 128), overlap=0.5, device="cuda", use_amp=True):
    """Fast sliding window inference."""
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd*(1-overlap)), int(ph*(1-overlap)), int(pw*(1-overlap))
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian blending
    gz = np.exp(-0.5*((np.arange(pd)-pd/2)/(pd*0.125))**2)
    gy = np.exp(-0.5*((np.arange(ph)-ph/2)/(ph*0.125))**2)
    gx = np.exp(-0.5*((np.arange(pw)-pw/2)/(pw*0.125))**2)
    gauss = (gz[:,None,None] * gy[None,:,None] * gx[None,None,:]).astype(np.float32)
    
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if D > pd and (D-pd) not in z_pos: z_pos.append(D-pd)
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if H > ph and (H-ph) not in y_pos: y_pos.append(H-ph)
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if W > pw and (W-pw) not in x_pos: x_pos.append(W-pw)
    
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = volume[z:z+pd, y:y+ph, x:x+pw].astype(np.float32)
                actual_shape = patch.shape
                
                if patch.shape != (pd, ph, pw):
                    pad = [(0, max(0, pd-patch.shape[0])), 
                           (0, max(0, ph-patch.shape[1])), 
                           (0, max(0, pw-patch.shape[2]))]
                    patch = np.pad(patch, pad, mode='reflect')
                
                patch = normalize_volume(patch)
                inp = torch.from_numpy(patch[None, None]).to(device)
                
                with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                    out = torch.sigmoid(model(inp)).squeeze().cpu().numpy()
                
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)


# =============================================================================
# RUN INFERENCE
# =============================================================================
test_df = pd.read_csv(cfg.TEST_CSV)
print(f"Found {len(test_df)} test volumes")

print("\n" + "="*70)
print("V3 INFERENCE (InstanceNorm3d)")
print("="*70)
print(f"Patch size: {cfg.PATCH_SIZE}")
print(f"Overlap: {cfg.OVERLAP}")
print(f"Threshold: {cfg.FINAL_THRESHOLD}")
print("="*70)

with zipfile.ZipFile(cfg.SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for idx, row in test_df.iterrows():
        image_id = row["id"]
        tif_path = cfg.TEST_IMAGES_DIR / f"{image_id}.tif"
        
        print(f"\n[{idx+1}/{len(test_df)}] Processing {image_id}...")
        volume = tifffile.imread(str(tif_path))
        print(f"  Volume shape: {volume.shape}")
        
        prob_map = sliding_window_inference(
            model=model,
            volume=volume,
            patch_size=cfg.PATCH_SIZE,
            overlap=cfg.OVERLAP,
            device=cfg.DEVICE,
            use_amp=cfg.USE_AMP
        )
        
        print(f"  Probability range: [{prob_map.min():.4f}, {prob_map.max():.4f}]")
        
        prediction = simplified_postprocess(
            prob_map,
            threshold=cfg.FINAL_THRESHOLD,
            min_component_size=cfg.MIN_COMPONENT_SIZE,
        )
        
        out_path = cfg.OUTPUT_DIR / f"{image_id}.tif"
        tifffile.imwrite(out_path, prediction.astype(np.uint8))
        zf.write(out_path, arcname=f"{image_id}.tif")
        out_path.unlink()
        
        del volume, prob_map, prediction
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n" + "="*70)
print(f"Submission created: {cfg.SUBMISSION_ZIP}")
print("="*70)